In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Define relative dataset path
file_path = os.path.join("data", "Sample - Superstore.csv")

# Load dataset
try:
    df = pd.read_csv(file_path, encoding='latin1')
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print("Error: The file was not found at the specified path.")
    

In [ ]:
df.shape


In [ ]:
# 1. DATA PREPROCESSING & INTEGRITY

# Standardize temporal fields to datetime objects
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Validate data integrity by removing null entries
# Note: Cleaning step ensures downstream analytical consistency
df = df.dropna()

# 2. FEATURE ENGINEERING

# Calculate Profit Margin via vectorized conditional logic
# np.where handles zero-division edge cases for Sales records
df['Profit Margin'] = np.where(df['Sales'] != 0, df['Profit'] / df['Sales'], 0)

# Validate transformation output
print(f"Preprocessing complete. Current dataset shape: {df.shape}")
display(df.head()) 


In [ ]:

# 3. MULTI-DIMENSIONAL BUSINESS ANALYSIS

# A) Product Performance: Identify primary profit drivers and negative profit products
# Aggregating total profit by product name for performance ranking
product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False)

print("Executive Summary: Top 5 High-Performing Products")
display(product_profit.head(5).to_frame())

print("\nExecutive Summary: Bottom 5 Under-Performing Products")
display(product_profit.tail(5).to_frame())

# B) Regional Evaluation: Comparative Sales vs. Profit metrics. Putting Sales and Profit side-by-side # Sorting by Profit to evaluate regional operational efficiency
regional_analysis = df.groupby('Region')[['Sales', 'Profit']].sum().sort_values(by='Profit', ascending=False)
print("\nRegional Operational Performance Metrics:")
display(regional_analysis)

# C) Discount Optimization: Correlation between discount rates and margins
# Calculating mean Profit Margin to assess the impact of discounting strategies
discount_analysis = df.groupby('Discount')['Profit Margin'].mean().reset_index()
print("\nMean Profit Margin by Discount Tier:")
display(discount_analysis.head(10))

# 4. DATA EXPORT & BI INTEGRATION

import os

# Verify current working directory for environment logging
print(f"Current System Directory: {os.getcwd()}")

# Export cleaned dataset via absolute path for downstream BI consumption
# Path ensures consistency across local and cloud-integrated environments
export_path = "/Users/mennatarek/Desktop/product-performance-intelligence/cleaned_superstore.csv"

try:
    df.to_csv(export_path, index=False)
    print(f"\nSUCCESS: Cleaned dataset exported to: {export_path}")
except Exception as e:
    print(f"Export Error: {e}")
    

In [ ]:
%pip install seaborn


In [ ]:

# 5. BUSINESS INTELLIGENCE VISUALIZATIONS
import matplotlib.pyplot as plt
import seaborn as sns

# Configure global visualization parameters for professional reporting
sns.set_theme(style="whitegrid")
plt.figure(figsize=(16, 12))

# A) Categorical Performance: Profitability by Sub-Category
# Purpose: Identify high-performing product segments for resource allocation
plt.subplot(2, 2, 1)
subcat_profit = df.groupby('Sub-Category')['Profit'].sum().sort_values(ascending=False)
sns.barplot(x=subcat_profit.values, y=subcat_profit.index, palette="viridis")
plt.title("Profit Distribution: Sub-Category Analysis", fontsize=14, fontweight='bold')
plt.xlabel("Total Profit ($)")
plt.ylabel("Sub-Category")

# B) Correlation Analysis: Impact of Discounting on Operational Margins
# Purpose: Visualize the 'Discount Trap' through scatter distribution
plt.subplot(2, 2, 2)
sns.scatterplot(data=df, x='Discount', y='Profit Margin', alpha=0.5, color='crimson')
plt.axhline(0, color='black', linestyle='--') # Baseline indicator for break-even point
plt.title("Margin Correlation: Discount Strategy Impact", fontsize=14, fontweight='bold')
plt.xlabel("Discount Rate")
plt.ylabel("Calculated Profit Margin")

# C) Market Composition: Regional Revenue Contribution
# Purpose: Assessment of market share distribution across active regions
plt.subplot(2, 2, 3)
region_sales = df.groupby('Region')['Sales'].sum()
plt.pie(region_sales, labels=region_sales.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette("pastel"))
plt.title("Regional Revenue Composition", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:

# 6. ADVANCED MARKETING & TEMPORAL ANALYSIS

# Convert text to dates so Python can plot them on a timeline
df['Order Date'] = pd.to_datetime(df['Order Date'])

plt.figure(figsize=(16, 6))

# A) Time-Series Analysis: Revenue and Profitability Seasonality
plt.subplot(1, 2, 1)

# Grouping data by month to see trends more clearly
monthly_trend = df.groupby(df['Order Date'].dt.to_period('M'))[['Sales', 'Profit']].sum().reset_index()
monthly_trend['Order Date'] = monthly_trend['Order Date'].dt.to_timestamp()

# Drawing two lines to compare total sales vs. total profit
sns.lineplot(data=monthly_trend, x='Order Date', y='Sales', label='Total Revenue', color='royalblue', linewidth=2)
sns.lineplot(data=monthly_trend, x='Order Date', y='Profit', label='Total Profit', color='seagreen', linewidth=2)

# Fill the area under the profit line to make it stand out
plt.fill_between(monthly_trend['Order Date'], monthly_trend['Profit'], color='seagreen', alpha=0.1)

plt.title("Temporal Trends: Revenue vs. Net Profit", fontsize=14, fontweight='bold')
plt.xlabel("Operational Timeline")
plt.ylabel("Financial Value ($)")
plt.legend()

# B) Segmental Contribution: Profitability by Customer Vertical
plt.subplot(1, 2, 2)

# Checking which type of customer (ex: Corporate) is giving us the most profit
segment_profit = df.groupby('Segment')['Profit'].sum().sort_values(ascending=False)
sns.barplot(x=segment_profit.index, y=segment_profit.values, palette="magma")

plt.title("Segmental Analysis: Total Profit Contribution", fontsize=14, fontweight='bold')
plt.xlabel("Market Segment")
plt.ylabel("Aggregated Profit ($)")

plt.tight_layout()
plt.show()